# Séance 12 — TP Épreuve : Pipeline Agent IA + MCP + Dashboard
### Cours "Visualisation de Données"

---

> **🎯 Objectif de l'épreuve**
> Construire de A à Z un pipeline end-to-end :
> deux serveurs MCP → agent ReAct → dashboard Plotly → interface Streamlit
> sur le dataset Airbnb Montréal.

---

## 📋 Consignes

- **Durée** : 3 heures — travail strictement individuel
- **Dataset fourni** : `listings_clean.csv` (Airbnb Montréal, déjà dans le dossier)
- **Rendu** : ce notebook + les fichiers `.py` générés, compressés en `.zip`
- **Barème** : 30 points répartis sur 6 étapes
- **Ollama doit être lancé** localement (`ollama serve` + modèle `llama3.2`)

---

## Prérequis techniques

```bash
pip install langchain langgraph langchain-openai langchain-mcp-adapters mcp duckdb streamlit nest-asyncio pandas plotly
```


In [ ]:
# ─── NE PAS MODIFIER — Setup imports ───
import subprocess, sys, os, json, re, webbrowser, asyncio
import pandas as pd, plotly.graph_objects as go
from plotly.subplots import make_subplots
import duckdb
import nest_asyncio; nest_asyncio.apply()

from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient

llm = ChatOpenAI(model='llama3.2', base_url='http://localhost:11434/v1', api_key='ollama', temperature=0)

async def show_steps(agent, question: str):
    """Affiche les étapes de raisonnement de l'agent."""
    print(f"► {question}")
    print("─" * 60)
    async for chunk in agent.astream({"messages": [("user", question)]}):
        if "agent" in chunk:
            msg = chunk["agent"]["messages"][0]
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  ▶ Tool : {tc['name']}({tc['args']})")
            elif msg.content:
                print(f"  ✓ {msg.content[:300]}")
        elif "tools" in chunk:
            obs = chunk["tools"]["messages"][0].content
            print(f"  ◀ Résultat : {str(obs)[:200]}")
    print("─" * 60)

print('✓ Setup OK — LLM configuré')
print(f'Répertoire : {os.getcwd()}')


---
# PARTIE 1 — RAPPELS EXPRESS
*Lire avant de commencer — 10 minutes*

---

## 💡 Rappel 1 — `Optional[float]` dans les serveurs MCP

Quand un LLM appelle un tool MCP, il peut passer `null` pour un paramètre numérique.
Sans `Optional`, Pydantic lève une erreur de validation. **Toujours déclarer** :

```python
from typing import Optional

@mcp.tool()
def get_weather(city: str, latitude: Optional[float] = None, longitude: Optional[float] = None) -> str:
    lat = latitude if latitude is not None else 45.51   # valeur par défaut explicite
    lon = longitude if longitude is not None else -73.55
    ...
```

> ⚠️ Si vous écrivez `latitude: float = 45.51`, Pydantic refusera `null` → l'agent plantera.


## 💡 Rappel 2 — `nest_asyncio` et `asyncio.run()` dans Streamlit

Streamlit tourne dans une boucle d'événements. Pour appeler du code `async` depuis un callback Streamlit :

```python
import nest_asyncio, asyncio
nest_asyncio.apply()          # à appeler UNE fois au démarrage

# Dans un callback Streamlit :
answer, steps = asyncio.run(run_agent(question))   # ✓ fonctionne avec nest_asyncio
```

> Sans `nest_asyncio.apply()`, `asyncio.run()` lève `RuntimeError: This event loop is already running`.


## 💡 Rappel 3 — `%%writefile` pour écrire des fichiers `.py`

La magic `%%writefile` écrit le contenu de la cellule directement dans un fichier,
sans problème d'échappement de caractères :

```python
%%writefile mcp_mon_serveur.py
from mcp.server.fastmcp import FastMCP
mcp = FastMCP('MonServeur')

@mcp.tool()
def mon_tool(param: str) -> str:
    return f"Résultat : {param}"

if __name__ == '__main__':
    mcp.run()
```

> Avantage : pas besoin de gérer les `\n`, `\'` ou `\"` dans une string Python.


## 💡 Rappel 4 — `MultiServerMCPClient` et tools locaux

```python
client = MultiServerMCPClient({
    'airbnb':  {'command': sys.executable, 'args': [os.path.abspath('mcp_airbnb_server.py')],  'transport': 'stdio'},
    'weather': {'command': sys.executable, 'args': [os.path.abspath('mcp_weather_server.py')], 'transport': 'stdio'},
})
mcp_tools = await client.get_tools()          # tools MCP
all_tools  = list(mcp_tools) + [make_dashboard]  # + tools locaux @tool
agent = create_react_agent(llm, all_tools, prompt=SYSTEM_PROMPT)
```

> Le `system_prompt` est **crucial** : sans règles explicites, le LLM peut oublier d'appeler `make_dashboard()`.


---
# PARTIE 2 — TP ÉPREUVE
*Complétez chaque étape dans les cellules prévues*

---

## Étape 1 — Serveur MCP Airbnb `mcp_airbnb_server.py` *(6 pts)*

Créez le fichier `mcp_airbnb_server.py` avec **5 tools** :

| Tool | Description | Points |
|------|-------------|--------|
| `describe_listings()` | Compte total, prix moyen, nb quartiers **+ liste des colonnes** | 1 pt |
| `query_data(sql)` | SQL libre sur la table `listings`, gestion d'erreur avec info colonnes | 1.5 pt |
| `top_neighbourhoods(n)` | Top N quartiers par note moyenne (note, prix moy, nb logements) | 1 pt |
| `price_stats(quartier)` | Statistiques de prix (moy, médiane, min, max) — global si quartier vide | 1 pt |
| `superhost_ratio()` | % de superhôtes par quartier — top 5 seulement | 1.5 pt |

**Colonnes disponibles dans `listings_clean.csv` :**
`id, name, quartier, latitude, longitude, room_type, price, minimum_nights, number_of_reviews, review_scores_rating, reviews_per_month`

> 💡 Utilisez `%%writefile` pour écrire le fichier directement.
> 💡 La connexion DuckDB doit être recréée à chaque appel : `conn = duckdb.connect()` + `CREATE TABLE IF NOT EXISTS listings AS SELECT * FROM read_csv_auto(...)`.


In [ ]:
%%writefile mcp_airbnb_server.py
# Votre code ici — remplacez chaque "TODO" par votre implémentation

from mcp.server.fastmcp import FastMCP
import duckdb, os

mcp = FastMCP('AirbnbServer')
CSV_PATH = os.path.join(os.path.dirname(__file__), 'listings_clean.csv')

def get_conn():
    # TODO : créer connexion DuckDB + charger listings_clean.csv
    # conn = duckdb.connect()
    # conn.execute(f"CREATE TABLE IF NOT EXISTS listings AS SELECT * FROM read_csv_auto('{CSV_PATH}')")
    # return conn
    raise NotImplementedError("Implémentez get_conn()")

@mcp.tool()
def describe_listings() -> str:
    """Retourne une vue générale du dataset Airbnb Montréal."""
    # TODO : utiliser get_conn() et retourner un résumé sous forme de string
    return "TODO: describe_listings() non implémenté"

@mcp.tool()
def query_data(sql: str) -> str:
    """Execute une requête SQL sur le dataset. Table disponible : listings.
    Colonnes: id, name, quartier, latitude, longitude, room_type, price,
    minimum_nights, number_of_reviews, review_scores_rating, reviews_per_month."""
    # TODO : exécuter sql avec get_conn() et retourner le résultat en string
    return "TODO: query_data() non implémenté"

@mcp.tool()
def top_neighbourhoods(n: int = 5) -> str:
    """Retourne les N meilleurs quartiers par note moyenne."""
    # TODO
    return "TODO: top_neighbourhoods() non implémenté"

@mcp.tool()
def price_stats(quartier: str = '') -> str:
    """Statistiques de prix. Si quartier est vide, stats globales."""
    # TODO
    return "TODO: price_stats() non implémenté"

@mcp.tool()
def superhost_ratio() -> str:
    """Retourne le % de superhôtes par quartier (top 5)."""
    # TODO
    return "TODO: superhost_ratio() non implémenté"

if __name__ == '__main__':
    mcp.run()

In [ ]:
# Test Étape 1 — Phase A : vérifier que le serveur démarre et expose les tools
# ✅ Ce test fonctionne MÊME avant d'implémenter les tools (les stubs retournent "TODO")
async def test_airbnb_tools():
    client = MultiServerMCPClient({
        'airbnb': {
            'command': sys.executable,
            'args': [os.path.abspath('mcp_airbnb_server.py')],
            'transport': 'stdio',
        }
    })
    tools = await client.get_tools()
    print(f'Tools Airbnb détectés ({len(tools)}) :')
    for t in tools:
        print(f'  ✓ {t.name} — {t.description[:60]}')
    attendus = {'describe_listings', 'query_data', 'top_neighbourhoods', 'price_stats', 'superhost_ratio'}
    manquants = attendus - {t.name for t in tools}
    if manquants:
        print(f'\n⚠️  Tools manquants : {manquants}')
    else:
        print('\n✅ Les 5 tools sont présents')

await test_airbnb_tools()

# ─────────────────────────────────────────────────────────────────
# Test Étape 1 — Phase B : appel réel (à lancer APRÈS implémentation)
# ⚠️ Ne lancez cette cellule qu'une fois describe_listings() implémenté
# ─────────────────────────────────────────────────────────────────
# async def test_airbnb_appel():
#     client = MultiServerMCPClient({
#         'airbnb': {'command': sys.executable, 'args': [os.path.abspath('mcp_airbnb_server.py')], 'transport': 'stdio'}
#     })
#     tools = await client.get_tools()
#     agent = create_react_agent(llm, tools)
#     result = await agent.ainvoke({"messages": [("user", "Décris le dataset en une phrase.")]})
#     print("Réponse :", result["messages"][-1].content[:300])
# await test_airbnb_appel()

## Étape 2 — Serveur MCP Météo `mcp_weather_server.py` *(4 pts)*

Créez le fichier `mcp_weather_server.py` avec **2 tools** :

| Tool | Description | Points |
|------|-------------|--------|
| `get_weather(city, latitude, longitude)` | Météo actuelle : temp actuelle, min/max, précipitations | 2 pts |
| `get_forecast_7days(city, latitude, longitude)` | Prévisions sur 7 jours : moy/min/max par jour | 2 pts |

> ⚠️ **Obligatoire** : utiliser `Optional[float]` pour `latitude` et `longitude` (valeurs par défaut : lat=45.51, lon=-73.55)
> 💡 API gratuite : `https://api.open-meteo.com/v1/forecast?latitude=...&longitude=...&hourly=temperature_2m,precipitation_probability&forecast_days=1`


In [ ]:
%%writefile mcp_weather_server.py
# Votre code ici — remplacez chaque "TODO" par votre implémentation

from mcp.server.fastmcp import FastMCP
from typing import Optional
import requests

mcp = FastMCP('WeatherServer')

@mcp.tool()
def get_weather(city: str, latitude: Optional[float] = None, longitude: Optional[float] = None) -> str:
    """Récupère la météo actuelle pour une ville via Open-Meteo."""
    # TODO : appeler l'API Open-Meteo et retourner temp actuelle, min/max, précipitations
    return "TODO: get_weather() non implémenté"

@mcp.tool()
def get_forecast_7days(city: str, latitude: Optional[float] = None, longitude: Optional[float] = None) -> str:
    """Retourne les prévisions de température sur 7 jours."""
    # TODO : appeler l'API Open-Meteo (forecast_days=7) et retourner moy/min/max par jour
    return "TODO: get_forecast_7days() non implémenté"

if __name__ == '__main__':
    mcp.run()

In [ ]:
# Test Étape 2 — Phase A : vérifier que le serveur météo démarre et expose les tools
# ✅ Ce test fonctionne MÊME avant d'implémenter les tools
async def test_weather_tools():
    client = MultiServerMCPClient({
        'weather': {
            'command': sys.executable,
            'args': [os.path.abspath('mcp_weather_server.py')],
            'transport': 'stdio',
        }
    })
    tools = await client.get_tools()
    print(f'Tools Météo détectés ({len(tools)}) :')
    for t in tools:
        print(f'  ✓ {t.name} — {t.description[:60]}')
    attendus = {'get_weather', 'get_forecast_7days'}
    manquants = attendus - {t.name for t in tools}
    if manquants:
        print(f'\n⚠️  Tools manquants : {manquants}')
    else:
        print('\n✅ Les 2 tools sont présents')

await test_weather_tools()

# ─────────────────────────────────────────────────────────────────
# Test Étape 2 — Phase B : appel réel (à lancer APRÈS implémentation)
# ⚠️ Ne lancez cette cellule qu'une fois get_weather() implémenté
# ─────────────────────────────────────────────────────────────────
# async def test_weather_appel():
#     client = MultiServerMCPClient({
#         'weather': {'command': sys.executable, 'args': [os.path.abspath('mcp_weather_server.py')], 'transport': 'stdio'}
#     })
#     tools = await client.get_tools()
#     agent = create_react_agent(llm, tools)
#     result = await agent.ainvoke({"messages": [("user", "Quel temps fait-il à Montréal ?")]})
#     print("Réponse :", result["messages"][-1].content[:300])
# await test_weather_appel()

## Étape 3 — Connexion des deux serveurs *(3 pts)*

Créez un `MultiServerMCPClient` qui connecte **simultanément** les deux serveurs
et listez l'ensemble des tools disponibles.

| Critère | Points |
|---------|--------|
| Les deux serveurs sont connectés en même temps | 1 pt |
| La liste complète des tools s'affiche (≥ 6 tools attendus) | 1 pt |
| Un appel de test réel à chaque serveur réussit | 1 pt |


In [ ]:
# Étape 3 — Combinez les deux serveurs et testez chaque tool
async def test_combined():
    # TODO : créer MultiServerMCPClient avec airbnb + weather
    # TODO : afficher la liste des tools
    # TODO : faire un appel de test (ex: describe_listings + get_weather)
    pass

await test_combined()


## Étape 4 — Dashboard Plotly `make_dashboard()` *(5 pts)*

Créez un tool `@tool` local nommé `make_dashboard(title, top_n)` qui génère
un dashboard HTML avec **4 figures en 2×2** :

| Figure | Type Plotly | Données | Points |
|--------|-------------|---------|--------|
| Fig 1 (1,1) | `Scattermapbox` | Carte des logements, colorée par prix | 1.5 pt |
| Fig 2 (1,2) | `Bar` horizontal | Prix moyen par quartier (top N) | 1 pt |
| Fig 3 (2,1) | `Box` | Distribution des prix par `room_type` | 1 pt |
| Fig 4 (2,2) | `Scatter` | Note vs Prix (scatter avec opacité 0.4) | 1 pt |
| Sauvegarde + ouverture navigateur | `fig.write_html()` + `webbrowser.open()` | — | 0.5 pt |

> 💡 `make_subplots(rows=2, cols=2, specs=[[{"type":"scattermapbox"}, {"type":"bar"}], [{"type":"box"}, {"type":"scatter"}]])`
> 💡 Mapbox style : `open-street-map`, centre : `lat=45.52, lon=-73.57`, zoom=10


In [ ]:
# Étape 4 — Créez le tool make_dashboard

@tool
def make_dashboard(title: str = 'Dashboard Montreal', top_n: int = 10) -> str:
    """Génère un dashboard HTML avec 4 figures Plotly (Airbnb Montréal).
    Args:
        title: Titre du dashboard
        top_n: Nombre de quartiers à afficher dans le bar chart
    """
    # TODO : lire listings_clean.csv avec pandas
    # TODO : créer make_subplots 2x2
    # TODO : Fig 1 - carte scattermapbox
    # TODO : Fig 2 - bar chart prix par quartier
    # TODO : Fig 3 - box plot par room_type
    # TODO : Fig 4 - scatter note vs prix
    # TODO : fig.update_layout() + write_html() + webbrowser.open()
    return "TODO: make_dashboard() non implémenté"

# Test direct (ne fonctionne qu'après implémentation)
result = make_dashboard.invoke({'title': 'Test Étape 4', 'top_n': 10})
print(result)

## Étape 5 — Agent end-to-end *(5 pts)*

Créez l'agent complet avec les deux serveurs MCP + le tool local `make_dashboard`.

| Critère | Points |
|---------|--------|
| `SYSTEM_PROMPT` définit l'ordre d'appel des tools (describe → top → weather → dashboard) | 1.5 pt |
| `run_dashboard_agent()` connecte les 2 serveurs + ajoute `make_dashboard` | 1.5 pt |
| Test avec le prompt "voyageur business budget 150$/nuit" → dashboard généré | 1 pt |
| Test avec un 2e prompt de votre choix (profil différent) → comportement observé | 1 pt |

> 💡 Le `SYSTEM_PROMPT` doit contenir des règles absolues : appelle TOUJOURS `make_dashboard()`,
> ne génère JAMAIS de code Python dans la réponse.


In [ ]:
# Étape 5a — Définissez le SYSTEM_PROMPT consultant BI
SYSTEM_PROMPT = """
# TODO : écrivez un system prompt qui :
# 1. Définit le rôle de l'agent (analyste BI tourisme Montréal)
# 2. Impose l'ordre d'appel des tools (describe → top_neighbourhoods → get_weather → make_dashboard)
# 3. Interdit de générer du code Python ou HTML
# 4. Impose de terminer par UNE phrase de synthèse
"""
print('✓ SYSTEM_PROMPT défini')


In [ ]:
# Étape 5b — Créez run_dashboard_agent()
async def run_dashboard_agent(question: str):
    """Lance l'agent complet avec les 2 serveurs MCP + make_dashboard."""
    # TODO : MultiServerMCPClient avec airbnb + weather
    # TODO : récupérer les tools MCP
    # TODO : combiner avec [make_dashboard]
    # TODO : create_react_agent + show_steps
    pass


In [ ]:
# Étape 5c — Test avec le prompt "voyageur business"
await run_dashboard_agent(
    "Je suis en voyage business à Montréal avec un budget max 150$/nuit. "
    "Génère un dashboard avec les quartiers adaptés et la météo."
)


In [ ]:
# Étape 5d — Test avec un 2e profil de votre choix
# Exemples possibles : famille avec enfants, étudiant budget serré, touriste culturel, groupe d'amis
# Observez : quels tools l'agent appelle-t-il ? Dans quel ordre ?

await run_dashboard_agent(
    "..."  # TODO : remplacez par votre prompt
)


## Étape 6 — Interface Streamlit `app.py` *(5 pts)*

Créez une application Streamlit complète en utilisant `%%writefile app.py`.

| Critère | Points |
|---------|--------|
| Sidebar avec au moins 2 filtres interactifs (prix, quartier...) | 1 pt |
| Dashboard Plotly affiché et mis à jour selon les filtres | 1.5 pt |
| Interface de chat (`st.chat_input`) qui appelle l'agent MCP | 1.5 pt |
| L'application se lance sans erreur (`python -m streamlit run app.py`) | 1 pt |

> 💡 Structure recommandée :
> ```python
> with st.sidebar:      # filtres
> fig, n = build_dashboard(filtres)
> st.plotly_chart(fig)
> question = st.chat_input("...")
> if question:
>     answer, steps = asyncio.run(run_agent(question))
> ```
> 💡 N'oubliez pas `nest_asyncio.apply()` en haut du fichier et `os.chdir(os.path.dirname(__file__))`.


In [ ]:
%%writefile app.py
# Étape 6 — Votre application Streamlit complète

import streamlit as st
import asyncio, os, sys, json, re
import pandas as pd, plotly.graph_objects as go
from plotly.subplots import make_subplots
import nest_asyncio; nest_asyncio.apply()

os.chdir(os.path.dirname(os.path.abspath(__file__)))

from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

st.set_page_config(page_title="Dashboard IA Montréal", page_icon="🏙️", layout="wide")
st.title("🏙️ Dashboard Airbnb Montréal — TP Épreuve")

llm = ChatOpenAI(model="llama3.2", base_url="http://localhost:11434/v1", api_key="ollama", temperature=0)

# TODO : ajouter les filtres en sidebar
# TODO : fonction build_dashboard(filtres) → Plotly figure
# TODO : fonction async run_agent(question) → (answer, steps)
# TODO : interface de chat


In [ ]:
# Lancez votre application Streamlit (décommentez)
# import subprocess
# subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'app.py'])
# print("✓ Streamlit lancé → http://localhost:8501")


---
## Synthèse *(2 pts)*

Complétez le tableau ci-dessous et répondez aux questions dans la cellule markdown suivante.


### Mon architecture

| Brique | Fichier | Tool(s) créé(s) |
|--------|---------|-----------------|
| Serveur MCP données | `mcp_airbnb_server.py` | *à compléter* |
| Serveur MCP API | `mcp_weather_server.py` | *à compléter* |
| Tool local | — | `make_dashboard` |
| Agent | — | `run_dashboard_agent` |
| Interface | `app.py` | — |

### Difficultés rencontrées
*Décrivez en 3-5 lignes les principales difficultés que vous avez rencontrées et comment vous les avez résolues.*

*(votre réponse ici)*

### Ce que j'ai appris
*En 2-3 lignes.*

*(votre réponse ici)*
